# Cubic Root Degeneracies and the Cusp Discriminant

This notebook explores the cubic thermal-mode dispersion relation through its **depressed-cubic form**

\[
x^3 + a x + b = 0,
\]

and studies how its roots change as the physical wavenumber \(k\) varies.

The discriminant

\[
\Delta = -4a^3-27b^2
\]

separates regions with different root structures:

- \(\Delta>0\): three distinct real roots;
- \(\Delta<0\): one real root and one complex-conjugate pair;
- \(\Delta=0\): at least two roots coincide.

The curve \(4a^3+27b^2=0\) is also the bifurcation set of the canonical cusp catastrophe. Here, however, \(a\) and \(b\) are not independent controls: both are functions of the single physical parameter \(k\). The calculation therefore traces a **one-dimensional path through the two-dimensional \((a,b)\) control plane**.

The notebook focuses first on the algebraic statement—root degeneracy of the dispersion relation. Interpreting a crossing as a dynamical saddle-node bifurcation requires an additional dynamical-system identification and is therefore kept separate from the numerical root analysis.

## 1. Imports

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Optional interactive 3D figure
try:
    import plotly.graph_objects as go
    from numpy.polynomial import Polynomial
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False

plt.rcParams.update({'font.size': 12})

## 2. Thermal-mode cubic and depressed-cubic transformation

The starting dispersion relation used in the original notebook is

\[
n^3+\frac{N_\rho}{t_{\rm cool}}n^2
+c_s^2k^2 n
+\frac{N_p}{\gamma t_{\rm cool}}c_s^2k^2=0,
\]

with

\[
N_\rho=N_{\rho0}+\left(\frac{k}{k_F}\right)^2,
\qquad
N_p=N_{p0}+\left(\frac{k}{k_F}\right)^2,
\]

and

\[
N_{p0}=\gamma R N_{\rho0}.
\]

For a general monic cubic

\[
n^3+B n^2+C n+D=0,
\]

the substitution

\[
x=n+\frac{B}{3}
\]

gives

\[
x^3+a x+b=0.
\]

We use \(a\) and \(b\) for the depressed-cubic coefficients to avoid confusing them with the coefficients of the original cubic.

In [ ]:
# Symbolic variables
n = sp.symbols('n')
gamma, R = sp.symbols('gamma R', real=True)
Nrho0, tcool, cs2, k, kF = sp.symbols(
    'Nrho0 tcool cs2 k kF', real=True
)

Np0 = gamma * R * Nrho0
Nrho = Nrho0 + (k/kF)**2
Np = Np0 + (k/kF)**2

B = Nrho / tcool
C = cs2 * k**2
D = (Np / (gamma * tcool)) * cs2 * k**2

dispersion = sp.expand(n**3 + B*n**2 + C*n + D)

# n = x - B/3, equivalently x = n + B/3
x = sp.symbols('x', real=True)
a_sym = sp.simplify(C - B**2/3)
b_sym = sp.simplify(2*B**3/27 - B*C/3 + D)

depressed = sp.expand(x**3 + a_sym*x + b_sym)

print("Original cubic:")
display(dispersion)

print("\nDepressed-cubic coefficient a(k):")
display(a_sym)

print("\nDepressed-cubic coefficient b(k):")
display(b_sym)

print("\nDepressed cubic:")
display(depressed)

## 3. Numerical parameter choice

The numerical values below are retained from the original exploratory notebook. They are dimensionless.

In particular, the original notebook fixes \(N_{\rho0}\) and obtains \(N_{p0}\) from \(R\),

\[
N_{p0}=\gamma R N_{\rho0}.
\]

These values are an illustrative parameter set for studying the cubic root structure; they should not be interpreted as a general physical classification.

In [ ]:
gamma_val = 5/3
R_val = 0.1
Nrho0_val = -10.0
Np0_val = R_val * gamma_val * Nrho0_val

lambdaF_val = 0.2
kF_val = 2*np.pi/lambdaF_val

tcool_val = 1.0
cs2_val = 1.0
cs_val = np.sqrt(cs2_val)

lambdath_val = cs_val * tcool_val
csi2_val = cs2_val / gamma_val
csi_val = np.sqrt(csi2_val)

print(f"Np0       = {Np0_val:.8f}")
print(f"Nrho0     = {Nrho0_val:.8f}")
print(f"R         = {R_val:.8f}")
print(f"lambda_F  = {lambdaF_val:.8f}")
print(f"k_F       = {kF_val:.8f}")
print(f"lambda_th = {lambdath_val:.8f}")
print(f"c_i       = {csi_val:.8f}")

## 4. Helper functions

The expressions for \(a(k)\) and \(b(k)\) are evaluated directly from the depressed cubic. The discriminant is

\[
\Delta(k)=-4a(k)^3-27b(k)^2.
\]

The function `lambda_critical()` below is retained from the original notebook as an independent polynomial calculation of the critical wavenumbers.

In [ ]:
def a_of_k(k):
    k = np.asarray(k)
    return (
        cs2_val*k**2
        - (Nrho0_val*kF_val**2 + k**2)**2
          / (3*kF_val**4*tcool_val**2)
    )


def b_of_k(k):
    k = np.asarray(k)
    return (
        Nrho0_val*R_val*cs2_val*k**2/tcool_val
        - Nrho0_val*cs2_val*k**2/(3*tcool_val)
        - cs2_val*k**4/(3*kF_val**2*tcool_val)
        + cs2_val*k**4/(gamma_val*kF_val**2*tcool_val)
        + 2*(Nrho0_val*kF_val**2 + k**2)**3
          /(27*kF_val**6*tcool_val**3)
    )


def discriminant(k):
    a = a_of_k(k)
    b = b_of_k(k)
    return -4*a**3 - 27*b**2


def lambda_critical():
    coefs = np.zeros(9)

    coefs[8] = -108 * gamma_val * cs2_val * Nrho0_val**4 * R_val

    term6 = (
        cs2_val*kF_val**2
        * (-1 + 9*R_val*(-2 + 3*R_val))
        * tcool_val**2 * gamma_val
        + 4*Nrho0_val*(1 + 3*R_val*gamma_val)
    )
    coefs[6] = -27*cs2_val*Nrho0_val**2/kF_val**2 * term6

    term4 = (
        2*gamma_val*cs_val**6*kF_val**4*tcool_val**4
        - cs_val**4*kF_val**2*Nrho0_val*tcool_val**2
          * (9 + 9*R_val*(gamma_val - 3) + gamma_val)
        + 6*cs2_val*Nrho0_val**2*(1 + gamma_val*R_val)
    )
    coefs[4] = -54/kF_val**4 * term4

    term2 = (
        -4*gamma_val*Nrho0_val*(3 + gamma_val*R_val)
        + cs2_val*kF_val**2*tcool_val**2
          * (-27 + gamma_val*(gamma_val + 18))
    )
    coefs[2] = 27*cs2_val/(gamma_val*kF_val**6) * term2

    coefs[0] = -108*cs2_val/kF_val**8

    return np.roots(coefs)

## 5. Field wavenumbers

The original notebook also evaluates the characteristic Field wavenumbers and the conductive cutoff for the condensation mode.

In [ ]:
kField_T = Nrho0_val / (tcool_val * cs_val)
kField_K = tcool_val * cs_val * kF_val**2
kField_rho = (Nrho0_val - Np0_val) / (tcool_val * cs_val)

print(f"k_T   = {kField_T:.8f}")
print(f"k_K   = {kField_K:.8f}")
print(f"k_rho = {kField_rho:.8f}")

if Np0_val < 0:
    kField_cond = np.sqrt(-Np0_val) * kF_val
    lambdaField_cond = 2*np.pi/kField_cond

    print(f"\nk_c (condensation) = {kField_cond:.8f}")
    print(
        "Field expression     = "
        f"{np.sqrt(kField_K*(kField_rho-kField_T)):.8f}"
    )
    print(f"lambda_c             = {lambdaField_cond:.8f}")

## 6. Locate the discriminant crossings

A zero of \(\Delta(k)\) marks a repeated root of the cubic. We first locate the crossings numerically and then compare them with the positive real roots returned by `lambda_critical()`.

In [ ]:
k_zoom = np.linspace(5.535, 5.63, 2000)
Delta_zoom = discriminant(k_zoom)

crossing_idx = np.where(
    np.signbit(Delta_zoom[:-1]) != np.signbit(Delta_zoom[1:])
)[0]

# Linear interpolation of each zero crossing
k_cross = []
for i in crossing_idx:
    k1, k2 = k_zoom[i], k_zoom[i+1]
    d1, d2 = Delta_zoom[i], Delta_zoom[i+1]
    kc = k1 - d1*(k2-k1)/(d2-d1)
    k_cross.append(kc)

k_cross = np.asarray(k_cross)

print("Discriminant crossings:")
for kc in k_cross:
    print(f"  k = {kc:.10f}, lambda = {2*np.pi/kc:.10f}")

kcrit_all = lambda_critical()
mask = (
    np.isclose(kcrit_all.imag, 0.0, atol=1e-10)
    & (kcrit_all.real > 0)
)
kcrit = np.sort(kcrit_all.real[mask])

print("\nPositive real roots from lambda_critical():")
for kc in kcrit:
    print(f"  k = {kc:.10f}, lambda = {2*np.pi/kc:.10f}")

## 7. Discriminant near the degeneracies

The sign of the discriminant gives the root structure of the depressed cubic:

\[
\Delta>0 \Rightarrow 3\ \text{distinct real roots},
\]

\[
\Delta<0 \Rightarrow 1\ \text{real root}+1\ \text{complex-conjugate pair}.
\]

At \(\Delta=0\), roots coincide.

In [ ]:
a_zoom = a_of_k(k_zoom)
b_zoom = b_of_k(k_zoom)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_zoom, Delta_zoom, label=r"$\Delta$")
ax.axhline(0, linewidth=0.8, linestyle="--")

for kc in k_cross:
    ax.axvline(kc, linewidth=0.8, linestyle=":")

ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$\Delta$")
ax.set_title("Cubic discriminant near the root degeneracies")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 8. Path through cusp control space

For the canonical cusp potential

\[
V(x;a,b)=\frac{x^4}{4}+\frac{a x^2}{2}+bx,
\]

stationary points satisfy

\[
\frac{\partial V}{\partial x}=x^3+ax+b=0.
\]

The repeated-root set is

\[
4a^3+27b^2=0.
\]

Our physical dispersion relation maps \(k\) to

\[
k\mapsto (a(k),b(k)),
\]

so varying \(k\) traces a one-dimensional path through this two-dimensional plane.

Crossing the dashed curve means that the **algebraic root structure of the cubic changes**. The catastrophe-theory interpretation is useful geometrically, but the eigenvalue roots of a dispersion relation should not automatically be identified with stable/unstable equilibria of a gradient dynamical system.

In [ ]:
# Use positive k because the coefficients depend on k^2.
k_path = np.linspace(0.0, 7.0, 2000)
a_path = a_of_k(k_path)
b_path = b_of_k(k_path)

# Cusp bifurcation set: b = +/- 2(-a/3)^(3/2), a <= 0
a_min = min(a_path.min(), -1e-6)
a_cusp = np.linspace(a_min, 0.0, 2000)
b_cusp = 2 * (-a_cusp/3)**1.5

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(a_cusp, b_cusp, "k--", label=r"$4a^3+27b^2=0$")
ax.plot(a_cusp, -b_cusp, "k--")

ax.plot(a_path, b_path, linewidth=2, label=r"$(a(k),b(k))$")

sample = np.linspace(0, len(k_path)-1, 12, dtype=int)
ax.scatter(a_path[sample], b_path[sample], s=25)

for kc in k_cross:
    ax.scatter(a_of_k(kc), b_of_k(kc), s=55, zorder=5)

ax.axhline(0, linewidth=0.5)
ax.axvline(0, linewidth=0.5)
ax.set_xlabel(r"$a$")
ax.set_ylabel(r"$b$")
ax.set_title("One-parameter path through cusp control space")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 9. Root branches of the depressed cubic

We now solve

\[
x^3+a(k)x+b(k)=0
\]

directly. Note that \(x\) is the **shifted eigenvalue variable**,

\[
x=n+\frac{N_\rho}{3t_{\rm cool}},
\]

not the original eigenvalue \(n\).

Therefore, after solving for \(x\), the physical dispersion-relation roots are recovered from

\[
n=x-\frac{N_\rho}{3t_{\rm cool}}.
\]

In [ ]:
def roots_x(k_value):
    return np.roots([1.0, 0.0, a_of_k(k_value), b_of_k(k_value)])


def shift_B(k_value):
    Nrho_value = Nrho0_val + (k_value/kF_val)**2
    return Nrho_value/tcool_val


def roots_n(k_value):
    return roots_x(k_value) - shift_B(k_value)/3


k_modes = np.linspace(5.535, 5.63, 700)

roots_x_all = np.array([roots_x(ki) for ki in k_modes])
roots_n_all = np.array([roots_n(ki) for ki in k_modes])

# For visualization only, sort independently by real part at each k.
order = np.argsort(roots_n_all.real, axis=1)
roots_sorted = np.take_along_axis(roots_n_all, order, axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
for j in range(3):
    ax.plot(k_modes, roots_sorted[:, j].real, linewidth=1)

for kc in k_cross:
    ax.axvline(kc, linestyle=":", linewidth=0.8)

ax.axhline(0, linewidth=0.6, linestyle="--")
ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$\mathrm{Re}(n)$")
ax.set_title("Real parts of the cubic eigenvalue roots")
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for j in range(3):
    ax.plot(k_modes, roots_sorted[:, j].imag, linewidth=1)

for kc in k_cross:
    ax.axvline(kc, linestyle=":", linewidth=0.8)

ax.axhline(0, linewidth=0.6, linestyle="--")
ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$\mathrm{Im}(n)$")
ax.set_title("Imaginary parts of the cubic eigenvalue roots")
ax.grid(alpha=0.3)
plt.show()

### A note on branch tracking

Sorting roots independently at every \(k\) is adequate for visualising the local root structure, but it is **not a rigorous mode-tracking algorithm**. Near degeneracies, labels can exchange.

For quantitative branch tracking, roots should instead be matched continuously between neighbouring \(k\)-values, for example by minimizing the distance in the complex plane or by using eigenfunction information when available.

## 10. The same root structure versus wavelength

Since

\[
\lambda=\frac{2\pi}{k},
\]

the local root structure can also be displayed as a function of wavelength. The wavelength array is sorted before plotting so that the horizontal coordinate increases from left to right.

In [ ]:
lam_modes = 2*np.pi/k_modes
sort_lam = np.argsort(lam_modes)

fig, ax = plt.subplots(figsize=(9, 5))
for j in range(3):
    ax.plot(
        lam_modes[sort_lam],
        roots_sorted[sort_lam, j].real,
        linewidth=1
    )

ax.axhline(0, linewidth=0.6, linestyle="--")
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(r"$\mathrm{Re}(n)$")
ax.set_title("Real parts of the cubic roots versus wavelength")
ax.grid(alpha=0.3)
plt.show()

## 11. Optional: cusp solution surface

The real solutions of

\[
x^3+ax+b=0
\]

form the familiar folded cusp surface in \((a,b,x)\) space. The interactive figure below is optional and requires Plotly.

This surface is a visualization of the canonical cubic's real roots. It is useful for seeing where the solution surface folds, but it should not by itself be taken as proof that the plasma dispersion relation undergoes a dynamical catastrophe in the strict sense.

In [ ]:
if HAVE_PLOTLY:
    # Keep the grid modest so the notebook remains responsive.
    a_grid = np.linspace(min(a_path)-1, 1.0, 100)
    b_extent = max(abs(b_path.min()), abs(b_path.max())) + 1
    b_grid = np.linspace(-b_extent, b_extent, 100)
    A, Bgrid = np.meshgrid(a_grid, b_grid)

    X1 = np.full_like(A, np.nan, dtype=float)
    X2 = np.full_like(A, np.nan, dtype=float)
    X3 = np.full_like(A, np.nan, dtype=float)

    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            roots = Polynomial([Bgrid[i, j], A[i, j], 0, 1]).roots()
            real_roots = roots[np.isclose(roots.imag, 0.0, atol=1e-9)].real
            real_roots.sort()

            if len(real_roots) == 1:
                X1[i, j] = real_roots[0]
            elif len(real_roots) == 3:
                X1[i, j], X2[i, j], X3[i, j] = real_roots

    fig = go.Figure()

    for X in (X1, X2, X3):
        fig.add_surface(
            x=A,
            y=Bgrid,
            z=X,
            opacity=0.65,
            showscale=False
        )

    # Add the physical k-path on the real solution sheets.
    kk = np.linspace(5.535, 5.63, 180)
    for ki in kk:
        xr = roots_x(ki)
        xr = xr[np.isclose(xr.imag, 0.0, atol=1e-9)].real
        if len(xr):
            fig.add_trace(
                go.Scatter3d(
                    x=np.full(len(xr), a_of_k(ki)),
                    y=np.full(len(xr), b_of_k(ki)),
                    z=xr,
                    mode='markers',
                    marker=dict(size=2),
                    showlegend=False
                )
            )

    fig.update_layout(
        scene=dict(
            xaxis_title='a',
            yaxis_title='b',
            zaxis_title='shifted root x'
        ),
        title='Real-root surface of x³ + ax + b = 0'
    )

    fig.show()
else:
    print("Plotly is not installed; skipping the interactive 3D figure.")

## 12. Interpretation

For this parameter set, varying \(k\) causes the path \((a(k),b(k))\) to cross the discriminant set twice. At each crossing,

\[
\Delta=0,
\]

so two roots of the cubic coincide. Between and outside those crossings, the number of real roots differs according to the sign of \(\Delta\).

This gives a useful **cusp-discriminant description of the cubic root topology**.

A stronger statement—such as identifying the crossings as physical saddle-node bifurcations of thermal modes—requires care. In canonical catastrophe theory, \(x\) is an equilibrium coordinate obtained from \(\partial V/\partial x=0\). Here the cubic is a dispersion relation for eigenvalues. The shared cubic geometry establishes an algebraic correspondence, but the dynamical interpretation must be justified separately.

This distinction is important if the analysis is later used in a paper or thesis.